# ITASORL - end-to-end experiments on Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/iLevyTate/ITASORL/blob/main/notebooks/colab_gpu.ipynb)

Runs the selected experiment profile via `scripts/run_e2e.py`, checkpoints as it
goes, mirrors progress to Google Drive (optional), and downloads a portable
`bundle.zip` when done. All the run/resume logic lives in the repo scripts, so
this notebook stays a thin shell that rarely needs to change.

**Research question.** Does a from-scratch survival agent *incidentally* encode which
world it lives in (authentic vs a surrogate whose drag differs), read out by a linear
probe but never rewarded? The pre-registered bar is **probe AUROC >= 0.65**.

### How to use
1. (Optional, to keep notebook edits) **File -> Save a copy in Drive**.
2. **Runtime -> Change runtime type -> GPU** (T4 / L4 / A100).
3. Pick a `RUN_PROFILE` in the config form below.
4. **Runtime -> Run all**. Read the printed `SUMMARY.md` at the bottom when it
   finishes; `bundle.zip` downloads automatically.

### Run profiles
| Profile | ~Time | What it runs |
|---------|-------|--------------|
| `quick` | ~25 min | smoke test (reduced scale) |
| `full` | ~4 hr | full replication: A + B + B-v2 (300 upd, 3 seeds) |
| `bv3_regime` | ~3.7 hr | **B-v3 threshold attempt** - per-episode drag *regime* (identifiable + policy-relevant) |
| `bv3_regime_n10` | ~11.5 hr | **B-v3 power extension** - fresh seeds 0-9, regime coupling; resume across sessions |
| `bv2_ceiling` | ~3.7 hr | B-v2 (ar1) capacity ceiling (`--sysid-aux`): what the trunk *can* encode |
| `bv3_ceiling` | ~3.7 hr | B-v3 capacity ceiling (`--drift-mode regime --sysid-aux`): is the 0.65 bar reachable |
| `b2_only` | ~3.7 hr | B-v2 only, 3 seeds |
| `b2_seed0` | ~75 min | B-v2 only, seed 0 (replication-gap diagnostic) |
| `experiments_no_b2` | ~15 min | everything except B-v2 |

### If the session crashes or disconnects
Progress is checkpointed after every (drift, seed) cell and mirrored to Drive
every ~5 minutes. Just reopen the notebook and **Runtime -> Run all** again:
`run_e2e.py --profile` finds the unfinished run for your profile (locally, or
on Drive after a fresh VM) and continues it - at most a few minutes repeat.


In [ ]:
# @title Configure { display-mode: "form" }
RUN_PROFILE = "bv3_regime"  # @param ["quick", "full", "bv3_regime", "bv3_regime_n10", "bv2_ceiling", "bv3_ceiling", "b2_only", "b2_seed0", "experiments_no_b2"]
BRANCH = "main"  # @param {type:"string"}
USE_DRIVE = True  # @param {type:"boolean"}
FORCE_FRESH = False  # @param {type:"boolean"}
RESUME_RUN_DIR = ""  # @param {type:"string"}

from pathlib import Path  # noqa: E402

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
    REPO_DIR = "/content/ITASORL"
except ImportError:
    IN_COLAB = False
    REPO_DIR = str(Path.cwd().resolve())
    if not (Path(REPO_DIR) / "scripts" / "run_e2e.py").is_file():
        REPO_DIR = str(Path(REPO_DIR).parent)

REPO_URL = "https://github.com/iLevyTate/ITASORL.git"
DRIVE_RESULTS = "/content/drive/MyDrive/ITASORL_results"
RESUME_RUN_DIR = RESUME_RUN_DIR.strip()
USE_DRIVE = USE_DRIVE and IN_COLAB

print(f"profile={RUN_PROFILE}  branch={BRANCH}  use_drive={USE_DRIVE}  force_fresh={FORCE_FRESH}")
if BRANCH != "main":
    print(f"WARNING: running branch {BRANCH!r}, not main. For testing unmerged "
          "code only; do not treat the results as replication runs.")


## 1. Set up (Drive, code, dependencies, GPU)

`USE_DRIVE=True` asks for Drive access once per fresh runtime (Google requires
the popup each new VM; the notebook can't skip it). It buys you crash-proof
runs: progress mirrors to `MyDrive/ITASORL_results` every ~5 min and a new VM
resumes automatically.

If you'd rather never see the popup, untick `USE_DRIVE`: everything still runs
and the bundle still downloads at the end, but a runtime crash loses progress
(resume then only works within the same VM).


In [ ]:
import os, subprocess, sys

def sh(cmd):
    print(f"$ {cmd}", flush=True)
    subprocess.run(cmd, shell=True, check=True)

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(DRIVE_RESULTS, exist_ok=True)

if IN_COLAB:
    if os.path.isdir(REPO_DIR):
        sh(f"cd {REPO_DIR} && git fetch origin && git checkout {BRANCH} && git pull origin {BRANCH}")
    else:
        sh(f"git clone --branch {BRANCH} --single-branch {REPO_URL} {REPO_DIR}")
else:
    print(f"Local mode: using existing checkout at {REPO_DIR}")
os.chdir(REPO_DIR)
sh("git log -1 --oneline")

sh(f"{sys.executable} -m pip install -q -r requirements-dev.txt")

import torch  # noqa: E402
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0),
          f"({torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB)")
else:
    print("No GPU detected. Runtime -> Change runtime type -> GPU, then re-run this cell.")
    print("CPU-only works but expB2 will be much slower.")


In [ ]:
# Keep-alive (Colab only): nudges the session so it stays awake during the
# long run. Leave the browser tab open.
if IN_COLAB:
    from IPython.display import display, Javascript

    display(Javascript("""
    (function () {
      const intervalMs = 60000;
      setInterval(() => {
        fetch('/gen_' + Date.now()).catch(() => {});
      }, intervalMs);
      console.log('ITASORL keep-alive active every', intervalMs / 1000, 's');
    })();
    """))
    print("Keep-alive enabled. Leave this tab open until the run finishes.")
else:
    print("Local mode: keep-alive not needed.")


## 2. Run the experiments

One command; everything else (profile flags, checkpointing, Drive mirroring,
auto-resume of an unfinished run) is handled inside `scripts/run_e2e.py`. The
printed `$ ...` line shows the exact invocation.

- `FORCE_FRESH` abandons any unfinished run and starts over.
- `RESUME_RUN_DIR` is an advanced override to continue a specific run directory.


In [ ]:
import os, subprocess, sys, time  # noqa: F811

cmd = [sys.executable, "scripts/run_e2e.py", "--profile", RUN_PROFILE]
if USE_DRIVE:
    cmd += ["--drive-sync", DRIVE_RESULTS]
if FORCE_FRESH:
    cmd += ["--force-fresh"]
if RESUME_RUN_DIR:
    cmd += ["--resume", RESUME_RUN_DIR]

env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"
print("$", " ".join(cmd), flush=True)
t0 = time.time()
rc = subprocess.run(cmd, cwd=REPO_DIR, env=env).returncode
print(f"Wall time: {(time.time() - t0) / 60:.1f} min")
if rc != 0:
    raise RuntimeError(
        f"Run exited with code {rc}. Progress is checkpointed - re-run this cell "
        "(or Runtime -> Run all after a disconnect) to continue where it left off.")


## 3. Read the results

Prints `SUMMARY.md`, compares B-v2 survival @ drift 0.45 to the canonical
artifacts, and (when the profile dumped recurrent states) recomputes the
world-identity probe under level vs dispersion features - the cheapest test of
whether world identity hides in a *volatility* signature the level probe discards.


In [ ]:
import subprocess, sys  # noqa: F811
from pathlib import Path

RUN_DIR = Path((Path(REPO_DIR) / "results" / "LATEST_RUN.txt").read_text().strip())
BUNDLE = RUN_DIR / "bundle.zip"
print("Run directory:", RUN_DIR)
print("\n" + "=" * 72)
print((RUN_DIR / "SUMMARY.md").read_text())
print("=" * 72)

_cmp = Path(REPO_DIR) / "scripts" / "compare_expB2_artifacts.py"
if _cmp.is_file():
    print("\nB-v2 artifact comparison (survival @ drift 0.45)")
    subprocess.run([sys.executable, str(_cmp), "--run", str(RUN_DIR)],
                   cwd=REPO_DIR, check=False)

STATES = RUN_DIR / "artifacts" / "states"
if STATES.is_dir():
    print("\nVariance & selectivity re-analysis (no GPU)")
    subprocess.run([sys.executable, "scripts/reanalyze_expB2_states.py", str(STATES)],
                   cwd=REPO_DIR, check=False)
else:
    print("\n(no dumped states in this run; variance/selectivity re-analysis skipped)")


In [ ]:
# Download the bundle. During the run everything was already mirrored to
# Drive (if USE_DRIVE), so no extra copying is needed here.
if not BUNDLE.is_file():
    raise FileNotFoundError(f"Missing bundle: {BUNDLE} - did the run cell finish?")

print("Bundle:", BUNDLE)
if USE_DRIVE:
    print("Drive mirror:", f"{DRIVE_RESULTS}/{RUN_DIR.name}/bundle.zip")

if IN_COLAB:
    from google.colab import files
    files.download(str(BUNDLE))
    print("Browser download started for bundle.zip")


## What's in the bundle

| File | Purpose |
|------|---------|
| `SUMMARY.md` | Plain English outcome. **Read this first.** |
| `status.json` | Live step + last line (updated during run) |
| `manifest.json` | Step timings, status, artifact index |
| `combined.log` | Full stdout (updated live during run) |
| `steps/*.json` | Parsed metrics (AUROC per drift, etc.) |
| `artifacts/` | Figures + `expB2_results.json` |
| `artifacts/cells/` | Per (drift, seed) checkpoints (resume granularity) |
| `artifacts/states/` | Dumped recurrent states for offline re-probing |

**While running (local):** `python scripts/watch_run.py --follow`

**While running (Colab + Drive):** open `MyDrive/ITASORL_results/<run>/combined.log`

Unzip locally and open `SUMMARY.md` to decide whether the organism encoded world identity.
